# 08 · El modelo supervisor

Un asistente RAG que nadie revisa empeora en silencio: los documentos cambian, los usuarios preguntan cosas que
nadie anticipó, un umbral que era correcto en marzo es erróneo en junio. El módulo 07 te dio una **batería** que
ejecutas a mano. Este módulo automatiza la otra mitad: un **supervisor**, un segundo modelo que lee cada
respuesta que da el asistente, decide si fue buena, dice *por qué* cuando no lo fue y devuelve los fallos como
arreglos de la documentación. Eso es **mejora continua** con una persona todavía al mando.

```
 los usuarios preguntan ─► asistente (recupera, responde, cita) ─► registro de turnos
                                                     │
                       cada noche: el supervisor juzga cada turno nuevo
                                                     │
        ┌────────────────────┬───────────────────────┴──────────────┐
   respuestas buenas    fallos de recuperación                observaciones de usuarios
   entran en el banco   "qué faltaba y dónde"                 juzgadas: documentación | proceso
   (se sirven sin       ─► issue ─► agente de docs ─► PR ─► una persona revisa y fusiona
    llamar al modelo)                       │
                                  índice reconstruido, batería repetida, mejora medida
```

**Aprenderás**
1. El trabajo del supervisor y su contrato de salida (un veredicto, fidelidad, citas, un diagnóstico de recuperación, lo que faltaba), validado con Pydantic
2. El prompt real del juez y un sustituto determinista que se ejecuta sin conexión
3. Un registro de turnos y la evaluación diaria que lo convierte en una lista de fallos de recuperación
4. El **banco de respuestas**: servir respuestas validadas sin llamar al modelo e invalidarlas cuando cambian los documentos
5. Observaciones de usuarios ("solicitar una corrección"): juzgadas y encaminadas después a arreglos de la documentación o al responsable
6. Cerrar el bucle sobre el corpus de Harbourview y **medir** la mejora con preguntas reservadas
7. Las cautelas: el juez puede equivocarse, calibración con etiquetas humanas (kappa de Cohen), sesgos, coste

## 1. El asistente y sus documentos

El asistente es el de los módulos anteriores: recuperación híbrida (BM25 y embeddings MiniLM fusionados con
Reciprocal Rank Fusion, esta vez con pesos iguales, un ajuste realista sin afinar), tres fragmentos numerados
entregados al paso de respuesta y una respuesta citada. Los documentos viven ahora en un diccionario `DOCS` que
puede **cambiar**: el índice se reconstruye de forma incremental cuando cambia un pasaje, volviendo a calcular
solo los embeddings de ese pasaje.

In [1]:
# El manual de Harbourview: 22 pasajes, ficticio, escrito para este curso
PASSAGES = [
    ["p01", "Safety / Access", "Technicians must hold a valid GWO Basic Safety Training certificate before boarding a crew transfer vessel. Certificates are checked at the harbour gate and expire after two years."],
    ["p02", "Safety / Weather limits", "Crew transfer vessels do not sail when the significant wave height exceeds 1.5 m or the wind speed at the harbour exceeds 20 m/s. The duty coordinator decides at 06:00 and may cancel later."],
    ["p03", "Safety / Lone working", "Lone working inside a nacelle is forbidden. Every task inside a turbine needs at least two people, and both must carry a personal locator beacon."],
    ["p04", "Turbines / Specification", "Each of the 40 turbines is rated at 8.0 MW with a rotor diameter of 164 m. Cut-in wind speed is 3 m/s, rated speed is 12.5 m/s and cut-out speed is 25 m/s."],
    ["p05", "Turbines / Naming", "Turbines are named HV01 to HV40, numbered from the western edge of the array. The offshore substation is OSS1, in the centre."],
    ["p06", "Maintenance / Schedule", "Every turbine gets a planned service once a year between April and September. The service takes two technicians about two days, plus weather delays."],
    ["p07", "Maintenance / Gearbox oil", "Gearbox oil is sampled every six months. If the particle count exceeds ISO 4406 code 18/16/13, a filter change is scheduled within 14 days."],
    ["p08", "Maintenance / Blade inspection", "Blades are inspected by drone every 12 months and after any lightning alarm. Leading-edge erosion above grade 3 is repaired in the next planned service."],
    ["p09", "Faults / Alarm levels", "Alarms have three levels. Level 1 is informational. Level 2 needs a remote check within 4 hours. Level 3 stops the turbine and needs a site visit within 24 hours when the weather allows."],
    ["p10", "Faults / Pitch system", "A pitch system fault (code PF-203) usually means a low accumulator pressure. Reset it remotely once. If it returns within 48 hours, send a crew to replace the hydraulic accumulator."],
    ["p11", "Faults / Yaw misalignment", "A yaw error above 8 degrees for more than 10 minutes raises alarm YW-110. Persistent errors point to a faulty wind vane: the vane is recalibrated, then replaced if the error remains."],
    ["p12", "Faults / Grid loss", "On loss of grid connection the turbines stop and the converters isolate. Restart is allowed only when the grid operator confirms voltage within 5% of nominal for 10 minutes."],
    ["p13", "Environment / Marine mammals", "Piling and heavy cable work pause when a marine mammal is sighted within 500 m. Work resumes after 30 minutes without a sighting."],
    ["p14", "Environment / Bird collision", "Between October and March, turbines HV01 to HV08 are curtailed at night during the peak migration periods announced by the ornithology team."],
    ["p15", "Environment / Waste", "Used oil, filters and composite offcuts are brought ashore and logged in the waste register. Nothing is discharged at sea."],
    ["p16", "Logistics / Spare parts", "The harbour warehouse keeps the critical spares: two pitch accumulators, four wind vanes and a spare converter module. Reorder when stock falls below one unit."],
    ["p17", "Logistics / Helicopter", "A helicopter is used only for medical evacuation and for urgent Level 3 visits when vessels cannot sail. Booking goes through the duty coordinator."],
    ["p18", "Contacts / Duty coordinator", "The duty coordinator is reachable 24 hours a day on the control room line. Out-of-hours safety incidents are reported there first."],
    ["p19", "Reporting / Incidents", "Any injury or near miss is reported within one hour and a written report is filed within 24 hours. Environmental incidents follow the same deadlines."],
    ["p20", "Reporting / Performance", "Availability is reported monthly. The target is 97% time-based availability, counting weather downtime as available."],
    ["p21", "Grid / Export", "Power is exported at 66 kV through two cables to the onshore substation. The export limit agreed with the grid operator is 300 MW."],
    ["p22", "Training / Induction", "New staff complete a two-day induction covering the handbook, emergency drills and the sea survival course before their first transfer."],
]

In [2]:
import re, math, json, time, hashlib, difflib, tempfile, os, textwrap
from collections import Counter
from typing import Literal, Optional
import numpy as np
from pydantic import BaseModel, Field, ValidationError
from sentence_transformers import SentenceTransformer

STOP = set("a an and are as at be by can do does for from how i if in is it of on or that the this to what when which who with you".split())
def tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [{} for _ in docs]
        for i, d in enumerate(docs):
            for t in tokens(d):
                self.tf[i][t] = self.tf[i].get(t, 0) + 1
        self.len = np.array([sum(tf.values()) for tf in self.tf], dtype=float); self.avg = self.len.mean()
        df = {}
        for tf in self.tf:
            for t in tf:
                df[t] = df.get(t, 0) + 1
        self.idf = {t: math.log(1 + (len(docs) - c + 0.5) / (c + 0.5)) for t, c in df.items()}
    def scores(self, query):
        s = np.zeros(len(self.tf))
        for t in set(tokens(query)):
            if t in self.idf:
                for i, tf in enumerate(self.tf):
                    f = tf.get(t, 0)
                    if f:
                        s[i] += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
EMB_CACHE, ENCODED = {}, [0]
def embed(text):
    if text not in EMB_CACHE:
        EMB_CACHE[text] = model.encode(text, normalize_embeddings=True); ENCODED[0] += 1
    return EMB_CACHE[text]

DOCS = {pid: [section, text] for pid, section, text in PASSAGES}      # la documentación, editable

class Index:
    def __init__(self, docs):
        self.ids = list(docs)
        self.docs = {k: list(v) for k, v in docs.items()}
        self.chunks = [f"{docs[i][0]}. {docs[i][1]}" for i in self.ids]
        self.bm25 = BM25(self.chunks)
        self.E = np.stack([embed(c) for c in self.chunks])       # solo se codifican los fragmentos no vistos antes
    def text_hash(self, pid):
        return hashlib.sha1(self.docs[pid][1].encode()).hexdigest()[:10]

def rrf_scores(index, question, weights=(1.0, 1.0), k=60):
    total = np.zeros(len(index.ids))
    for sc, w in zip((index.bm25.scores(question), index.E @ embed(question)), weights):
        for rank, i in enumerate(np.argsort(-sc), start=1):
            total[i] += w / (k + rank)
    return total

def best_sentence(question, text):
    # las notas de vocabulario ("Also called: ...") ayudan a recuperar pero nunca se citan como respuesta
    q = set(tokens(question))
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text) if not s.startswith("Also called:")]
    return max(sentences, key=lambda s: len(q & set(tokens(s))))

PRICE_IN, PRICE_OUT = 0.20, 0.80      # USD ilustrativos por millón de tokens de un modelo alojado pequeño; consulta los precios actuales
def assistant(index, question, k=3):
    s_b, s_s = index.bm25.scores(question), index.E @ embed(question)
    order = np.argsort(-rrf_scores(index, question))[:k]
    pieces = [dict(n=n, pid=index.ids[i], section=index.docs[index.ids[i]][0], text=index.docs[index.ids[i]][1]) for n, i in enumerate(order, 1)]
    refused = bool(s_b.max() < 1.0 and s_s.max() < 0.30)
    answer = "I cannot find that in the handbook." if refused else f"{best_sentence(question, pieces[0]['text'])} [1]"
    t_in = int(1.3 * len(" ".join(p["text"] for p in pieces).split()) + 60)       # prompt: instrucciones + fragmentos + pregunta
    t_out = int(1.3 * len(answer.split()))
    return dict(question=question, pieces=pieces, answer=answer, refused=refused, model="answer-model-v1",
                tokens_in=t_in, tokens_out=t_out, cost_usd=(t_in * PRICE_IN + t_out * PRICE_OUT) / 1e6)

index = Index(DOCS)
t = assistant(index, "What wave height stops the crew boats from sailing?")
print(t["answer"], "| pieces:", [p["pid"] for p in t["pieces"]], f"| {t['tokens_in']} + {t['tokens_out']} tokens, ${t['cost_usd']:.6f}")

/home/patito/Developments/academy/rag-systems/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2363.09it/s]

Crew transfer vessels do not sail when the significant wave height exceeds 1.5 m or the wind speed at the harbour exceeds 20 m/s. [1] | pieces: ['p02', 'p01', 'p09'] | 184 + 32 tokens, $0.000062


## 2. El contrato del supervisor

El juez lee **lo que el asistente vio y dijo**: la pregunta, los fragmentos numerados, la respuesta (y el
resultado de una consulta de datos, si la hubo, como en el módulo 06). Devuelve un pequeño objeto JSON. Los
campos están elegidos para que cada uno impulse una acción posterior:

| Campo | Significado | Impulsa |
|---|---|---|
| `verdict` | `correct`, `partial`, `incorrect` o `no_answer` | la curva de calidad del día |
| `faithfulness` | de 1 a 5: ¿está cada afirmación y cada cifra en los fragmentos? | el banco de respuestas, alarmas de alucinación |
| `citations_ok` | ¿existe cada `[n]` y respalda su frase? | el banco de respuestas |
| `retrieval` | `yes`, `partial`, `no`: ¿contenían los fragmentos lo que la pregunta necesitaba? | **dónde mirar**: recuperación o generación |
| `missing` | cuando la recuperación no es `yes`: qué faltaba y dónde debería estar | el arreglo de la documentación |
| `sql_ok` | sin consulta, o una consulta que respondió a la pregunta sin error | el camino de datos (módulo 06) |
| `notes` | una frase de razonamiento | la persona que audita al juez |

Lo validamos con **Pydantic**, porque el juez es un modelo y el JSON de un modelo no puede darse por bueno a
ciegas: una puntuación de 7, un veredicto escrito `mostly ok` o un campo ausente deben detectarse en la
frontera, no tres pasos después.

In [3]:
class JudgeVerdict(BaseModel):
    verdict: Literal["correct", "partial", "incorrect", "no_answer"]
    faithfulness: int = Field(ge=1, le=5)
    citations_ok: bool
    retrieval: Literal["yes", "partial", "no"]
    missing: str = ""
    sql_ok: Optional[bool] = None
    notes: str = ""

good = '{"verdict": "correct", "faithfulness": 5, "citations_ok": true, "retrieval": "yes", "missing": "", "sql_ok": null, "notes": "ok"}'
bad = '{"verdict": "mostly ok", "faithfulness": 7, "citations_ok": "yes", "retrieval": "yes"}'
print("valid  :", JudgeVerdict.model_validate_json(good).verdict)
try:
    JudgeVerdict.model_validate_json(bad)
except ValidationError as e:
    for err in e.errors():
        print("invalid:", err["loc"][0], "->", err["msg"][:70])

valid  : correct
invalid: verdict -> Input should be 'correct', 'partial', 'incorrect' or 'no_answer'
invalid: faithfulness -> Input should be less than or equal to 5


## 3. El prompt real y un sustituto que se ejecuta sin conexión

Esta es la forma del prompt de juez que se usa en la práctica. Fíjate en lo que pide: **rigor** (el juez no debe
dar el beneficio de la duda), un **vocabulario cerrado** y `missing`, el campo que hace útil todo el bucle.

In [4]:
JUDGE_PROMPT = '''You evaluate the answers of a RAG assistant that answers only from numbered PIECES (and, if present,
a QUERY RESULT cited as [data]). Every sentence of a good answer ends with a citation [n] or [data], and when the
pieces do not contain the answer it says "I cannot find that in the handbook."
Judge strictly:
- verdict: "correct" (answers what was asked and everything is supported), "partial" (answers only part, or lacks
  precision), "incorrect" (states something the pieces contradict or do not support), "no_answer" (says it cannot find it).
- faithfulness: 1 to 5. 5 = every claim and figure is in the pieces, copied exactly.
- citations_ok: true only if every [n] exists and supports its sentence.
- retrieval: "yes" if the pieces had what the question needed, "partial" or "no".
- missing: if retrieval is not "yes", what information was absent and which page or topic should hold it; else "".
- sql_ok: true if there was no query, or the query answered the question without error.
- notes: one sentence with the reason for the verdict.
Reply with ONLY the JSON object {{"verdict": ..., "faithfulness": ..., "citations_ok": ..., "retrieval": ..., "missing": ..., "sql_ok": ..., "notes": ...}}.

PIECES:
{pieces}

QUESTION: {question}

ANSWER: {answer}'''
turn = assistant(index, "What qualification do I need before getting on the boat?")
print(JUDGE_PROMPT.format(pieces="\n".join(f"[{p['n']}] {p['text']}" for p in turn["pieces"]), question=turn["question"], answer=turn["answer"])[-760:])

 "notes": ...}.

PIECES:
[1] New staff complete a two-day induction covering the handbook, emergency drills and the sea survival course before their first transfer.
[2] Technicians must hold a valid GWO Basic Safety Training certificate before boarding a crew transfer vessel. Certificates are checked at the harbour gate and expire after two years.
[3] Crew transfer vessels do not sail when the significant wave height exceeds 1.5 m or the wind speed at the harbour exceeds 20 m/s. The duty coordinator decides at 06:00 and may cancel later.

QUESTION: What qualification do I need before getting on the boat?

ANSWER: New staff complete a two-day induction covering the handbook, emergency drills and the sea survival course before their first transfer. [1]


En producción el juez es un **modelo pequeño y barato** (un modelo de clase Haiku, o Amazon Nova Micro o Lite;
el más barato que concuerde bien con tus etiquetas humanas, ver sección 8) llamado a través de la misma interfaz
LiteLLM del módulo 04. Un juez con capacidad de razonamiento gasta la mayoría de sus tokens pensando: en el
sistema real del que sale este bucle, una llamada al juez usaba unos **1.500 tokens de razonamiento por
respuesta**, por lo que el presupuesto de tokens de la llamada debe ser generoso (un límite de 500 devolvió un
JSON vacío).

El notebook ejecutado **no llama a ningún modelo**. El sustituto de abajo es un juez determinista de *reglas más
embeddings*. Es deliberadamente más débil que un modelo de lenguaje, y mediremos exactamente cuánto más débil
(sección 8). Usa un truco que merece mención: para el campo `retrieval` pide una **segunda opinión**
ordenando todo el corpus solo con el modelo de embeddings y comprueba si ese pasaje estaba entre los fragmentos
que usó el asistente.

In [5]:
NUM = re.compile(r"\d+(?:\.\d+)?")

def second_opinion(index, question):
    sims = index.E @ embed(question)
    best = int(np.argmax(sims))
    return index.ids[best], float(sims[best])

def stand_in_judge(turn, index):
    pieces, answer, q = turn["pieces"], turn["answer"], turn["question"]
    vocab = {w for c in index.chunks for w in tokens(c)}
    best_pid, best_sim = second_opinion(index, q)
    pids = [p["pid"] for p in pieces]
    if turn["refused"]:
        needed = best_sim >= 0.45            # existe algo claramente relevante, así que negarse fue un fallo de recuperación
        retrieval = "no" if needed else "yes"
        return dict(verdict="no_answer", faithfulness=5, citations_ok=True, retrieval=retrieval, sql_ok=None,
                    missing=(f"the handbook has material close to the question in '{index.docs[best_pid][0]}'" if needed else ""),
                    notes="unnecessary refusal" if needed else "a correct refusal: nothing relevant in the handbook")
    cites = [int(c) for c in re.findall(r"\[(\d+)\]", answer)]
    body = re.sub(r"\s*\[\d+\]", "", answer)
    valid = bool(cites) and all(1 <= c <= len(pieces) for c in cites)
    cited_text = " ".join(pieces[c - 1]["text"] for c in cites) if valid else ""
    support = len(set(tokens(body)) & set(tokens(cited_text))) / max(len(set(tokens(body))), 1)
    numbers_ok = all(n in NUM.findall(cited_text) for n in NUM.findall(body))
    citations_ok = valid and support >= 0.6
    faith = 5 if citations_ok and numbers_ok else (3 if valid and numbers_ok else (2 if valid else 1))
    retrieval = "yes" if pids[0] == best_pid else ("partial" if best_pid in pids else "no")
    addresses = float(embed(body) @ embed(q)) >= 0.40
    if retrieval == "no":
        verdict = "incorrect"
    elif retrieval == "partial" or not addresses:
        verdict = "partial"
    else:
        verdict = "correct" if citations_ok else "partial"
    missing = ""
    if retrieval != "yes":
        unknown = [w for w in dict.fromkeys(tokens(q)) if w not in vocab]
        missing = (f"the pieces lack the passage '{index.docs[best_pid][0]}' ({best_pid}); words of the question absent from "
                   f"the whole handbook: {', '.join(unknown) or 'none'}")
    return dict(verdict=verdict, faithfulness=faith, citations_ok=citations_ok, retrieval=retrieval, missing=missing, sql_ok=None,
                notes=f"support {support:.0%}, numbers {'ok' if numbers_ok else 'NOT in the cited piece'}, answer-question similarity {'ok' if addresses else 'low'}")

def judge(turn, index, judge_fn=stand_in_judge, retries=1):
    # llamar al juez, validar su JSON, volver a preguntar una vez si la respuesta está mal formada
    for attempt in range(retries + 1):
        raw = json.dumps(judge_fn(turn, index))
        try:
            return JudgeVerdict.model_validate_json(raw)
        except ValidationError:
            continue
    raise ValueError("the judge never returned a valid verdict")

for q in ["What wave height stops the crew boats from sailing?", "What qualification do I need before getting on the boat?", "Who is the chief executive of the operating company?"]:
    v = judge(assistant(index, q), index)
    print(f"{q[:56]:56s} -> {v.verdict:9s} faith {v.faithfulness} retrieval {v.retrieval:7s} | {v.missing[:70] or v.notes[:70]}")

What wave height stops the crew boats from sailing?      -> correct   faith 5 retrieval yes     | support 100%, numbers ok, answer-question similarity ok
What qualification do I need before getting on the boat? -> partial   faith 5 retrieval partial | the pieces lack the passage 'Safety / Weather limits' (p02); words of 


Who is the chief executive of the operating company?     -> no_answer faith 5 retrieval yes     | a correct refusal: nothing relevant in the handbook


## 4. El registro de turnos y la evaluación diaria

Cada pregunta que responde el asistente se guarda como un **turno**: la pregunta, los fragmentos exactos que
leyó (su texto, para poder juzgar la respuesta más tarde aunque los documentos cambien), la respuesta, el
modelo, los tokens y el coste. Este registro es la materia prima de todo lo que sigue. Un trabajo nocturno juzga
los turnos aún no evaluados y escribe un **registro de calidad** por turno.

Simulamos un día de tráfico con las 20 preguntas del "Día 1" (el conjunto 1 del módulo 07, tratado aquí como
preguntas anónimas de usuarios: **el supervisor nunca ve las respuestas de referencia**).

In [6]:
# (id, pregunta, pasaje correcto o None, patrones de datos requeridos, patrones prohibidos, tipo)
SET1 = [
    ("s01", "What wave height stops the crew boats from sailing?", "p02", [r"1\.5 ?m"], [], "direct"),
    ("s02", "How often is the gearbox oil sampled?", "p07", [r"six months"], [], "direct"),
    ("s03", "Who may work alone inside a nacelle?", "p03", [r"forbidden"], [], "paraphrase"),
    ("s04", "What is the rotor diameter of a turbine?", "p04", [r"164 ?m"], [], "direct"),
    ("s05", "What do I do if the pitch fault comes back after a reset?", "p10", [r"replace the hydraulic accumulator"], [], "direct"),
    ("s06", "When must a Level 3 alarm be attended on site?", "p09", [r"24 hours"], [], "direct"),
    ("s07", "How soon must an injury be reported?", "p19", [r"one hour"], [], "direct"),
    ("s08", "What is the largest amount of power the farm is allowed to send ashore?", "p21", [r"300 ?MW"], [r"8\.0 ?MW"], "paraphrase"),
    ("s09", "Which turbines are switched off at night in the autumn to protect birds?", "p14", [r"HV01 to HV08"], [], "paraphrase"),
    ("s10", "How many spare wind vanes are kept in stock?", "p16", [r"four wind vanes"], [], "direct"),
    ("s11", "What qualification do I need before getting on the boat?", "p01", [r"GWO Basic Safety Training"], [], "paraphrase"),
    ("s12", "When can the turbines restart after a blackout?", "p12", [r"10 minutes"], [], "paraphrase"),
    ("s13", "When can a helicopter be used?", "p17", [r"medical evacuation"], [], "direct"),
    ("s14", "What happens when a dolphin is seen near the piling work?", "p13", [r"pause"], [], "paraphrase"),
    ("s15", "What is the price of electricity per MWh?", None, [], [], "trap"),
    ("s16", "Who is the chief executive of the operating company?", None, [], [], "trap"),
    ("s17", "What is the rated power of turbine HV99?", None, [], [], "trap"),
    ("s18", "What availability did the farm achieve last month?", None, [], [], "trap"),
    ("s19", "How many turbines does the farm have?", "p04", [r"40 turbines"], [], "direct"),
    ("s20", "What is the cut-in wind speed?", "p04", [r"3 ?m/s"], [], "direct"),
]
SET2 = [
    ("t01", "Are boats cancelled when the sea is too rough?", "p02", [r"1\.5 ?m"], [], "paraphrase"),
    ("t02", "Can I go up the tower alone to fix something?", "p03", [r"forbidden"], [], "paraphrase"),
    ("t03", "How big are the blades?", "p04", [r"164 ?m"], [], "paraphrase"),
    ("t04", "Who do I call in the middle of the night about an accident?", "p18", [r"control room line"], [], "paraphrase"),
    ("t05", "What if the oil in the gearbox gets dirty?", "p07", [r"filter change"], [], "paraphrase"),
    ("t06", "Do you stop work when whales or seals swim close?", "p13", [r"pause"], [], "paraphrase"),
    ("t07", "How much spare equipment is kept on shore?", "p16", [r"two pitch accumulators"], [], "paraphrase"),
    ("t08", "Is the farm allowed to produce at full output?", "p21", [r"300 ?MW"], [], "paraphrase"),
    ("t09", "What should new colleagues do before their first trip?", "p22", [r"induction"], [], "paraphrase"),
    ("t10", "How should rubbish and old oil be handled?", "p15", [r"brought ashore"], [], "paraphrase"),
    ("t11", "What does code PF-203 mean?", "p10", [r"low accumulator pressure"], [], "identifier"),
    ("t12", "alarm YW-110", "p11", [r"8 degrees"], [], "identifier"),
    ("t13", "ISO 4406 18/16/13", "p07", [r"filter change"], [], "identifier"),
    ("t14", "Where is OSS1?", "p05", [r"centre"], [], "identifier"),
    ("t15", "66 kV cables", "p21", [r"two cables"], [], "identifier"),
    ("t16", "What is the stock price of the operating company?", None, [], [], "trap"),
    ("t17", "What is the rotor diameter of turbine HV77?", None, [], [], "trap"),
    ("t18", "What was the monthly availability in July?", None, [], [], "trap"),
    ("t19", "How many people work on the farm?", None, [], [], "trap"),
    ("t20", "What is the cut-out wind speed?", None if False else "p04", [r"25 ?m/s"], [], "direct"),
]
BATTERY = SET1 + SET2
LOG_DIR = tempfile.mkdtemp(prefix="rag08_")
def append_jsonl(name, record):
    with open(os.path.join(LOG_DIR, name), "a") as f:
        f.write(json.dumps(record) + "\n")

def serve_day(index, battery, day):
    turns = []
    for n, q in enumerate(battery, 1):
        t = assistant(index, q[1])
        t.update(id=f"{day}_{n:02d}", day=day, first_turn=True)
        append_jsonl("chat_turns.jsonl", t)
        turns.append(t)
    return turns

day1 = serve_day(index, SET1, "2026-11-17")
print(len(day1), "turns logged;", f"{sum(t['tokens_in'] + t['tokens_out'] for t in day1)} tokens, ${sum(t['cost_usd'] for t in day1):.5f} for the answers")
print(open(os.path.join(LOG_DIR, "chat_turns.jsonl")).readline()[:230], "...")

20 turns logged; 3709 tokens, $0.00099 for the answers
{"question": "What wave height stops the crew boats from sailing?", "pieces": [{"n": 1, "pid": "p02", "section": "Safety / Weather limits", "text": "Crew transfer vessels do not sail when the significant wave height exceeds 1.5 m  ...


In [7]:
def evaluate_day(turns, index):
    records = []
    for t in turns:
        v = judge(t, index)
        rec = dict(turn_id=t["id"], question=t["question"], **v.model_dump())
        append_jsonl("chat_quality.jsonl", rec)
        records.append(rec)
    return records

def summarise(records):
    c = Counter(r["verdict"] for r in records)
    print("verdicts:", dict(c))
    print(f"mean faithfulness {np.mean([r['faithfulness'] for r in records]):.2f}, citations ok {np.mean([r['citations_ok'] for r in records]):.0%}, "
          f"retrieval yes {np.mean([r['retrieval'] == 'yes' for r in records]):.0%}")
    misses = [r for r in records if r["retrieval"] != "yes"]
    print(f"\n{len(misses)} retrieval misses (the input for improving the documentation):")
    for r in misses:
        print(f"  [{r['retrieval']:7s}] {r['question'][:64]}\n            missing: {r['missing'][:150]}")

quality1 = evaluate_day(day1, index)
summarise(quality1)

verdicts: {'correct': 16, 'partial': 2, 'incorrect': 1, 'no_answer': 1}
mean faithfulness 5.00, citations ok 100%, retrieval yes 85%

3 retrieval misses (the input for improving the documentation):
  [partial] What is the largest amount of power the farm is allowed to send 
            missing: the pieces lack the passage 'Grid / Export' (p21); words of the question absent from the whole handbook: largest, amount, farm
  [partial] What qualification do I need before getting on the boat?
            missing: the pieces lack the passage 'Safety / Weather limits' (p02); words of the question absent from the whole handbook: qualification, need, getting, boat
  [no     ] What is the price of electricity per MWh?
            missing: the pieces lack the passage 'Grid / Export' (p21); words of the question absent from the whole handbook: price, electricity, per, mwh


Este resumen es el objetivo de todo el ejercicio. No dice "el asistente es 70 % bueno": es una **lista de
trabajo**. Cada fallo de recuperación viene con *lo que faltaba*. La pregunta de la cualificación tiene
palabras que no existen en ningún lugar del manual (*boat* donde el manual dice *vessel*, *qualification*
donde dice *certificate*), lo que sugiere que el arreglo es de vocabulario, no un pasaje nuevo ni un modelo
nuevo. La pregunta de la exportación lista solo palabras genéricas como ausentes: el juez encontró la página
correcta pero no supo nombrar la laguna real (*send ashore* frente a *export*). La tercera es una **falsa
alarma** (una pregunta que el manual no puede responder, que el asistente rechazó correctamente, marcada como
fallo): un juez no siempre acierta, y por eso una persona sigue leyendo esta lista. Una persona que leyera el
registro en bruto encontraría lo mismo, una vez, con esfuerzo; el supervisor lo encuentra **cada noche**, para
cada pregunta, por céntimos.

## 5. El banco de respuestas

Si una pregunta ya se respondió **correctamente**, volver a responderla con el modelo es un gasto y un riesgo.
El **banco de respuestas** guarda las respuestas validadas y las devuelve, sin llamar al modelo, a preguntas casi
idénticas. Su regla de admisión es estricta, porque se confía en el banco:

* una pregunta de **primer turno** (no un seguimiento que dependa de la conversación),
* veredicto `correct`, fidelidad de al menos 4, citas correctas, recuperación `yes` y ninguna consulta fallida.

Guarda el **hash de cada fragmento en el que se apoyó la respuesta**. Cuando un documento cambia, los hashes ya
no coinciden y la entrada **deja de usarse**: una respuesta validada para el documento antiguo no está validada
para el nuevo.

In [8]:
class Bank:
    def __init__(self, threshold):
        self.threshold, self.entries = threshold, []
    def admit(self, turn, rec, index):
        ok = (turn["first_turn"] and rec["verdict"] == "correct" and rec["faithfulness"] >= 4 and rec["citations_ok"]
              and rec["retrieval"] == "yes" and rec["sql_ok"] in (None, True))
        if ok:
            cited = [turn["pieces"][int(n) - 1]["pid"] for n in re.findall(r"\[(\d+)\]", turn["answer"])]
            self.entries.append(dict(question=turn["question"], vec=embed(turn["question"]), answer=turn["answer"],
                                     relied_on={pid: index.text_hash(pid) for pid in cited}, origin="evaluation"))
        return ok
    def lookup(self, question, index):
        if not self.entries:
            return None, 0.0
        sims = [float(e["vec"] @ embed(question)) for e in self.entries]
        i = int(np.argmax(sims))
        e = self.entries[i]
        if sims[i] < self.threshold:
            return None, sims[i]
        if any(pid not in index.docs or index.text_hash(pid) != h for pid, h in e["relied_on"].items()):
            return None, sims[i]            # un fragmento en el que se apoyaba ha cambiado: invalidada
        return e, sims[i]

# ¿cuánto se parecen las preguntas casi idénticas y cuánto las preguntas DISTINTAS que lo parecen?
base = "What wave height stops the crew boats from sailing?"
probes = [("same question, one word dropped", "What wave height stops the crew boats sailing?"),
          ("rephrased, same meaning", "What is the wave limit for crew boats?"),
          ("looks similar, different fact (wind, not waves)", "What wind speed stops the crew boats from sailing?"),
          ("different topic", "How often is the gearbox oil sampled?")]
for label, p in probes:
    print(f"cosine {float(embed(base) @ embed(p)):.2f}  {label:50s} {p}")

cosine 1.00  same question, one word dropped                    What wave height stops the crew boats sailing?
cosine 0.83  rephrased, same meaning                            What is the wave limit for crew boats?
cosine 0.78  looks similar, different fact (wind, not waves)    What wind speed stops the crew boats from sailing?
cosine 0.09  different topic                                    How often is the gearbox oil sampled?


Esta tabla es como se elige el umbral, y muestra el peligro. *"What **wind speed** stops the boats"* es una
**pregunta distinta** con una respuesta distinta (el manual indica tanto un límite de oleaje como uno de
viento), y sin embargo su coseno con la original (0,78) está solo un poco por debajo del de una reformulación
verdadera (0,83). El margen entre "mismo significado" y "se parece pero responde distinto" es de unas pocas
centésimas, así que un umbral elegido para captar reformulaciones queda pegado a los parecidos. Un umbral
estricto de **0,95** solo sirve repeticiones casi literales, que es el valor seguro por defecto para un banco:
pierde algo de ahorro y nunca sirve una respuesta errónea a una pregunta parecida. Empieza estricto, afloja solo
con evidencia.

In [9]:
bank = Bank(threshold=0.95)
admitted = [bank.admit(t, r, index) for t, r in zip(day1, quality1)]
print(f"{sum(admitted)} of {len(day1)} answers entered the bank (the others failed one of the admission checks)")

for label, p in [("verbatim repeat", base), ("one word dropped", probes[0][1]), ("look-alike (wind speed)", probes[2][1])]:
    e, sim = bank.lookup(p, index)
    print(f"{label:26s} cosine {sim:.2f} -> {'BANK HIT, no model call: ' + e['answer'][:60] if e else 'no hit, the assistant answers'}")

# un documento cambia: la entrada que se apoyaba en p02 debe dejar de usarse
edited = {k: list(v) for k, v in DOCS.items()}
edited["p02"][1] += " Boats may also be called crew boats."
index_edited = Index(edited)
e, sim = bank.lookup(base, index_edited)
print("after p02 changed ->", "still served (BUG)" if e else "invalidated: the answer is recomputed from the new document")

16 of 20 answers entered the bank (the others failed one of the admission checks)
verbatim repeat            cosine 1.00 -> BANK HIT, no model call: Crew transfer vessels do not sail when the significant wave 
one word dropped           cosine 1.00 -> BANK HIT, no model call: Crew transfer vessels do not sail when the significant wave 
look-alike (wind speed)    cosine 0.78 -> no hit, the assistant answers
after p02 changed -> invalidated: the answer is recomputed from the new document


## 6. Observaciones de usuarios: "solicitar una corrección de esta respuesta"

Los usuarios ven errores que el juez no ve. La interfaz ofrece un botón de **solicitar corrección** junto a cada
respuesta, y la observación va a un **segundo juez** con una pregunta distinta: *¿está fundada la observación y,
si lo está, qué tipo de arreglo necesita?*

* `founded`: `yes`, `partial` o `no`. La documentación es la fuente de verdad **salvo que la observación
  demuestre que está mal o incompleta**, y al juez se le dice que no dé la razón por defecto (un usuario
  educado también puede equivocarse).
* `kind`: `documentation` (corregir o completar una página, o la página está bien pero el chat no pudo
  encontrarla), `process` (debe cambiar *la regla en sí*: un cálculo, un parámetro, un procedimiento) o `none`.

Los dos tipos siguen caminos distintos, y este es el diseño de seguridad:

* una observación de **documentación** abre un **issue** que un **agente de documentación** nocturno convierte en
  un **pull request**, que **una persona revisa y fusiona**. El agente solo edita archivos de documentación,
  nunca código, nunca el índice, y nunca fusiona;
* una observación de **proceso** es una decisión sobre cómo funciona el negocio. **Espera al responsable**, que
  la acepta o la rechaza. Ningún modelo cambia una regla de seguridad.

In [10]:
UNIT_PAIR = re.compile(r"(\d+(?:\.\d+)?)\s*(hours?|minutes?|months?|MW|kV|m)\b", re.I)
PROCESS_CUE = re.compile(r"\b(change the (rule|procedure|process|limit|calculation)|should depend|new rule|recalculate)\b", re.I)

def judge_remark(turn, comment, index):
    corpus = " ".join(index.docs[p][1] for p in index.ids)
    if PROCESS_CUE.search(comment):
        return dict(founded="partial", kind="process", analysis="The remark asks to change the rule itself; only the owner can decide that.", correction="")
    unsupported = [m for m in UNIT_PAIR.findall(comment) if not re.search(rf"\b{re.escape(m[0])}\s*{m[1]}", corpus, re.I)]
    if unsupported:
        return dict(founded="no", kind="none", analysis=f"The figure {' '.join(unsupported[0])} appears nowhere in the handbook; the documentation says otherwise.", correction="")
    pid, sim = second_opinion(index, turn["question"] + " " + comment)
    return dict(founded="yes", kind="documentation", pid=pid,
                analysis=f"The remark is supported by '{index.docs[pid][0]}' ({pid}), which the assistant did not use.",
                correction=f"Make '{index.docs[pid][0]}' easier to find for questions phrased like: {turn['question']}")

REMARKS = [
    ("What is the largest amount of power the farm is allowed to send ashore?", "The export limit is 300 MW, but you answered with a different topic."),
    ("How soon must an injury be reported?", "The handbook should say 2 hours, not one hour."),
    ("What wave height stops the crew boats from sailing?", "We should change the procedure: the limit should depend on the vessel size."),
]
ISSUES, by_question = [], {t["question"]: t for t in day1}
for question, comment in REMARKS:
    verdict = judge_remark(by_question[question], comment, index)
    print(f"remark: {comment}\n   -> founded={verdict['founded']}, kind={verdict['kind']}: {verdict['analysis']}")
    if verdict["kind"] == "documentation":
        ISSUES.append(dict(label="correccion-docs", pid=verdict["pid"], question=question, comment=comment, correction=verdict["correction"], state="open"))
print()
print("ISSUE OPENED (label correccion-docs):")
print(textwrap.indent(f"Passage: {ISSUES[0]['pid']}\nUser remark: {ISSUES[0]['comment']}\nQuestion: {ISSUES[0]['question']}\nProposed correction: {ISSUES[0]['correction']}", "  "))
print("\nPROCESS remark: status 'to decide', shown to the owner with Accept / Reject. Nothing changes until they choose.")

remark: The export limit is 300 MW, but you answered with a different topic.
   -> founded=yes, kind=documentation: The remark is supported by 'Grid / Export' (p21), which the assistant did not use.
remark: The handbook should say 2 hours, not one hour.
   -> founded=no, kind=none: The figure 2 hours appears nowhere in the handbook; the documentation says otherwise.
remark: We should change the procedure: the limit should depend on the vessel size.
   -> founded=partial, kind=process: The remark asks to change the rule itself; only the owner can decide that.

ISSUE OPENED (label correccion-docs):
  Passage: p21
  User remark: The export limit is 300 MW, but you answered with a different topic.
  Question: What is the largest amount of power the farm is allowed to send ashore?
  Proposed correction: Make 'Grid / Export' easier to find for questions phrased like: What is the largest amount of power the farm is allowed to send ashore?

PROCESS remark: status 'to decide', shown to the owne

## 7. Cerrar el bucle sobre el manual de Harbourview

Ahora el ciclo completo, con una **medición**. La vara de medir **no** es el juez: es la batería de referencia
escrita por personas del módulo 07 (pasaje correcto y datos requeridos), así que no calificamos al supervisor
consigo mismo.

1. Se sirve y se juzga el tráfico del **Día 1** (conjunto 1). Los fallos de recuperación se convierten en issues.
2. El **agente de documentación** propone la edición más pequeña por issue (una línea de vocabulario añadida a la
   página correcta) como pull request. Solo toca documentación.
3. **Una persona revisa el PR**. Rechaza, redirige y completa lo que propuso el agente. Esta es la persona en el
   bucle, y es lo que evita que las ediciones de vocabulario se conviertan en ruido.
4. Con la aprobación cambian las páginas y el **índice se reconstruye de forma incremental**.
5. El tráfico del **Día 2** (conjunto 2, preguntas *no usadas* para hacer los arreglos) se sirve con el índice
   antiguo y con el nuevo, y comparamos.

In [11]:
def reference_correct(q, turn):
    qid, question, gold, req, forbid, kind = q
    if gold is None:
        return turn["refused"]
    if turn["refused"]:
        return False
    cited = turn["pieces"][0]["pid"]
    return cited == gold and all(re.search(p, turn["answer"], re.I) for p in req) and not any(re.search(p, turn["answer"], re.I) for p in forbid)

def issues_from_misses(turns, records, index):
    out = []
    for t, r in zip(turns, records):
        if r["retrieval"] == "yes" or not r["missing"]:
            continue
        pid = re.search(r"\((p\d+)\)", r["missing"])
        words = re.search(r"absent from the whole handbook: (.*)$", r["missing"])
        terms = [w.strip() for w in words.group(1).split(",")] if words and words.group(1) != "none" else []
        if pid and terms:
            out.append(dict(pid=pid.group(1), terms=terms, question=t["question"]))
    return out

def docs_agent(issues, docs):
    # agente nocturno: la edición de documentación más pequeña por issue, nunca código, nunca el índice, nunca fusionado
    allowed = set(docs)                                  # solo puede tocar páginas de documentación
    edits = {}
    for i in issues:
        if i["pid"] in allowed:
            edits.setdefault(i["pid"], []).extend(i["terms"])
    pr = dict(branch="docs/corrections-2026-11-17", status="awaiting human review", changes=[])
    for pid, terms in edits.items():
        terms = list(dict.fromkeys(terms))
        new = docs[pid][1] + f" Also called: {', '.join(terms)}."
        pr["changes"].append(dict(pid=pid, terms=terms, old=docs[pid][1], new=new))
    return pr

misses = issues_from_misses(day1, quality1, index)
pr = docs_agent(misses, DOCS)
print(f"{len(misses)} issues -> pull request '{pr['branch']}' ({pr['status']}), {len(pr['changes'])} pages edited\n")
for ch in pr["changes"]:
    print(f"--- {ch['pid']} ({DOCS[ch['pid']][0]})   proposed terms: {ch['terms']}")

3 issues -> pull request 'docs/corrections-2026-11-17' (awaiting human review), 2 pages edited

--- p21 (Grid / Export)   proposed terms: ['largest', 'amount', 'farm', 'price', 'electricity', 'per', 'mwh']
--- p02 (Safety / Weather limits)   proposed terms: ['qualification', 'need', 'getting', 'boat']


Lee la propuesta del agente como lo hace el revisor, porque es **errónea de tres maneras**, y eso es normal:

* para `p21` propone palabras genéricas (*largest*, *amount*, *farm*) más *price, electricity, per, mwh*, que
  vienen de una pregunta que el manual **no debe** responder: la falsa alarma del juez se coló en una edición de
  documentación. Añadirlas haría que el pasaje de exportación coincidiera con preguntas sobre el precio de la
  electricidad;
* para la pregunta de la cualificación propone `p02` (la página del tiempo) porque las palabras sobre el barco
  apuntan allí, pero la página que realmente la responde es `p01`, **Safety / Access**, que habla de un
  *certificate*;
* no puede saber que el manual dice *vessel* en todos los lugares donde una tripulación diría *boat*.

Una persona arregla las tres. La revisión se escribe como datos para que puedas verla: cada cambio del agente
recibe una decisión y un motivo, y las ediciones aprobadas son las que se aplican.

In [12]:
REVIEW_DECISIONS = [
    ("p21 agent terms: largest, amount, farm, price, electricity, per, mwh", "REJECT", "generic words, and 'price/electricity' come from a question that must stay unanswerable (judge false alarm)"),
    ("p21 (user remark about 'send ashore')", "EDIT", "add the user's own words: 'send ashore'"),
    ("p02 agent terms: qualification, need, getting, boat", "REDIRECT", "the question is answered by p01 (Safety / Access, the certificate): move to p01, drop 'need' and 'getting'"),
    ("p02 vocabulary 'boat'", "ADD", "the handbook says 'vessel' on p01 and p02; crews say 'boat'. Note it on both pages (reviewer's domain knowledge)"),
]
APPROVED_EDITS = {"p01": ["qualification", "boat", "boats"], "p02": ["boat", "boats"], "p21": ["send ashore"]}

print(f"{'agent change or proposal':66s}{'decision':>10s}")
for what, decision, why in REVIEW_DECISIONS:
    print(f"{what:66s}{decision:>10s}   {why}")

NEW_DOCS = {k: list(v) for k, v in DOCS.items()}
print()
for pid, terms in APPROVED_EDITS.items():
    NEW_DOCS[pid][1] = DOCS[pid][1] + f" Also called: {', '.join(terms)}."
    diff = [l for l in difflib.unified_diff([DOCS[pid][1]], [NEW_DOCS[pid][1]], lineterm="", n=0) if l.startswith("+") and not l.startswith("+++")]
    print(f"{pid}: ... {diff[0][-60:]}")
before = ENCODED[0]
t0 = time.time()
new_index = Index(NEW_DOCS)
print(f"\nindex rebuilt in {time.time() - t0:.2f} s: {ENCODED[0] - before} of {len(NEW_DOCS)} passages re-embedded (the unchanged ones were reused)")

agent change or proposal                                            decision
p21 agent terms: largest, amount, farm, price, electricity, per, mwh    REJECT   generic words, and 'price/electricity' come from a question that must stay unanswerable (judge false alarm)
p21 (user remark about 'send ashore')                                   EDIT   add the user's own words: 'send ashore'
p02 agent terms: qualification, need, getting, boat                 REDIRECT   the question is answered by p01 (Safety / Access, the certificate): move to p01, drop 'need' and 'getting'
p02 vocabulary 'boat'                                                    ADD   the handbook says 'vessel' on p01 and p02; crews say 'boat'. Note it on both pages (reviewer's domain knowledge)

p01: ... re after two years. Also called: qualification, boat, boats.
p02: ... des at 06:00 and may cancel later. Also called: boat, boats.
p21: ...  with the grid operator is 300 MW. Also called: send ashore.



index rebuilt in 0.15 s: 3 of 22 passages re-embedded (the unchanged ones were reused)


In [13]:
def run_day(idx, battery):
    turns = [assistant(idx, q[1]) for q in battery]
    verdicts = [judge(t, idx) for t in turns]
    ref = [reference_correct(q, t) for q, t in zip(battery, turns)]
    return turns, verdicts, ref

print(f"{'':34s}{'reference correct':>20s}{'judge says correct':>21s}{'retrieval yes':>16s}")
results = {}
for label, battery in [("Day 1 (used to make the fixes)", SET1), ("Day 2 (held out)", SET2)]:
    for name, idx in [("before", index), ("after", new_index)]:
        turns, verdicts, ref = run_day(idx, battery)
        results[(label, name)] = (turns, verdicts, ref)
        print(f"{label:28s}{name:>6s}{sum(ref):>14d} / {len(ref)}{sum(v.verdict == 'correct' for v in verdicts):>16d} / {len(ref)}{sum(v.retrieval == 'yes' for v in verdicts):>11d} / {len(ref)}")

                                     reference correct   judge says correct   retrieval yes


Day 1 (used to make the fixes)before            13 / 20              16 / 20         17 / 20


Day 1 (used to make the fixes) after            14 / 20              18 / 20         19 / 20


Day 2 (held out)            before            10 / 20              11 / 20         13 / 20
Day 2 (held out)             after            10 / 20              11 / 20         13 / 20


Lee esta tabla en tres pasos.

1. **Día 1 antes y después** muestra los arreglos funcionando sobre las preguntas que los originaron. Es
   necesario pero halagador: escribimos los arreglos mirando esas mismas preguntas.
2. El **Día 2** es la prueba que cuenta, porque ninguna de sus preguntas se usó para hacer un arreglo. Compara
   sus filas de *antes* y *después*: eso es cuánto de la reparación fue vocabulario real y cuánto fue memorizar
   el Día 1.
3. Compara la columna de **referencia** con la columna del **juez**. El supervisor estima la calidad de cada
   pregunta sin ninguna etiqueta, que es su valor, y la distancia entre su estimación y la referencia es su
   error, que es la siguiente sección.

Aquí el informe honesto es corto. En el **Día 1** el recuento de referencia pasa de 13 a 14 correctas y la
recuperación `yes` de 17 a 19: la pregunta de la cualificación queda arreglada, y la pregunta del límite de
exportación ahora recupera el pasaje correcto pero sigue citando la frase equivocada de este (un problema de
generación, que ninguna edición de documentación toca). En el **Día 2** **no se mueve nada**: 13 de 20 con
recuperación `yes` y 10 de 20 correctas, antes y después. Las notas de vocabulario arreglan las palabras que
contienen, y los fallos del Día 2 usan *otras* palabras (blades, rubbish, "full output", "in the middle of the
night") o son trampas. No lo leas como un fracaso del bucle. Los arreglos de una noche son estrechos por
construcción, cuestan un pull request revisado y la batería muestra que no causaron ninguna regresión. La lista
de fallos de recuperación de mañana, construida con las preguntas del Día 2, es el siguiente pull request. Fíjate
también en la columna del juez: acredita 18 de 20 en el Día 1 donde la referencia dice 14, la distancia que se
mide en la sección 8.

## 8. Cautelas: el juez también es un modelo

Un supervisor que no has medido es una segunda fuente de errores con tono seguro. Cuatro hábitos lo mantienen
honesto.

**1. Calibra con etiquetas humanas.** Toma una muestra de respuestas, etiquétalas tú y compara. La exactitud
bruta halaga a un juez cuando la mayoría de las respuestas son correctas, así que calcula también el **kappa de
Cohen**: el acuerdo corregido por el acuerdo que se obtendría por azar (0 es azar, 1 es perfecto, por encima de
0,6 suele considerarse sustancial). Aquí las etiquetas humanas son la batería de referencia del módulo 07.

In [14]:
turns_b = results[("Day 1 (used to make the fixes)", "before")][0] + results[("Day 2 (held out)", "before")][0]
verd_b = results[("Day 1 (used to make the fixes)", "before")][1] + results[("Day 2 (held out)", "before")][1]
ref_b = results[("Day 1 (used to make the fixes)", "before")][2] + results[("Day 2 (held out)", "before")][2]
judge_b = [v.verdict == "correct" for v in verd_b]

def cohen_kappa(a, b):
    a, b = np.array(a, dtype=int), np.array(b, dtype=int)
    po = (a == b).mean()
    pe = sum((a == k).mean() * (b == k).mean() for k in (0, 1))
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

tp = sum(j and r for j, r in zip(judge_b, ref_b)); fp = sum(j and not r for j, r in zip(judge_b, ref_b))
fn = sum((not j) and r for j, r in zip(judge_b, ref_b)); tn = sum((not j) and (not r) for j, r in zip(judge_b, ref_b))
print(f"judge vs human reference on {len(ref_b)} answers:  both correct {tp}, both not {tn}, judge too generous {fp}, judge too harsh {fn}")
print(f"agreement {(tp + tn) / len(ref_b):.0%}, Cohen's kappa {cohen_kappa(judge_b, ref_b):.2f}")
print("\nwhere it is too generous (says correct, a human says not):")
for q, t, v, j, r in zip(SET1 + SET2, turns_b, verd_b, judge_b, ref_b):
    if j and not r:
        print(f"  {q[0]} ({q[5]:9s}) {q[1][:60]}")

judge vs human reference on 40 answers:  both correct 20, both not 10, judge too generous 7, judge too harsh 3
agreement 75%, Cohen's kappa 0.47

where it is too generous (says correct, a human says not):
  s05 (direct   ) What do I do if the pitch fault comes back after a reset?
  s12 (paraphrase) When can the turbines restart after a blackout?
  s17 (trap     ) What is the rated power of turbine HV99?
  s18 (trap     ) What availability did the farm achieve last month?
  t05 (paraphrase) What if the oil in the gearbox gets dirty?
  t17 (trap     ) What is the rotor diameter of turbine HV77?
  t18 (trap     ) What was the monthly availability in July?


Lee la lista de *demasiado generoso*: son los puntos ciegos del supervisor, y son informativos. Un sustituto de
reglas y embeddings no puede saber que una turbina **HV99** no existe, ni que un **objetivo** indicado no es un
valor alcanzado, y aprueba respuestas que citan la frase equivocada del pasaje correcto, porque ve un pasaje
relevante y aprueba. Un modelo de lenguaje real lee la frase y detecta muchos de estos casos, pero no todos, y
solo sabes cuáles ejecutando la calibración anterior **con el juez real**, sobre etiquetas escritas por una
persona. Mantén una muestra permanente de respuestas que una persona vuelve a etiquetar cada semana.

**2. Mantén a una persona en el bucle donde importa.** Fíjate en lo que se permitió hacer al supervisor en este
módulo: juzgar, ordenar, proponer issues. **No** se le permitió editar la documentación (el agente propone, una
persona fusiona), cambiar una regla (decide el responsable) ni promover una respuesta al banco sin pasar la
estricta prueba de admisión. La autonomía debe crecer con la confianza medida, no con el entusiasmo.

**3. No dejes que un modelo se califique a sí mismo.** Un juez tiende a preferir las respuestas escritas por su
propia familia de modelos (*sesgo de autopreferencia*) y a compartir sus puntos ciegos. Usa un **modelo
distinto** para el juez que para las respuestas, y comprueba el juez con personas (hábito 1) cada vez que
cambies cualquiera de los dos.

**4. Sabe lo que cuesta.** El juez lee la pregunta, los fragmentos y la respuesta, y después razona. Cuéntalo.

In [15]:
JUDGE_IN, JUDGE_OUT = 600, 1500          # tokens por respuesta juzgada: prompt más fragmentos de entrada, ~1.500 tokens de razonamiento de salida
per_answer = (JUDGE_IN * PRICE_IN + JUDGE_OUT * PRICE_OUT) / 1e6
answer_cost = np.mean([t["cost_usd"] for t in day1])
print(f"answering a question:  ${answer_cost:.5f} on average (illustrative small-model prices)")
print(f"judging an answer:     ${per_answer:.5f} ({JUDGE_IN} in + {JUDGE_OUT} out tokens): {per_answer / answer_cost:.0f}x the cost of the answer itself")
for per_day in (50, 200, 1000):
    print(f"{per_day:5d} answers a day: ${per_day * per_answer:6.2f} a day, ${per_day * per_answer * 30:7.2f} a month for judging every one")
print("with the answer bank serving repeats: the model is not called, and the judge is not either")

answering a question:  $0.00005 on average (illustrative small-model prices)
judging an answer:     $0.00132 (600 in + 1500 out tokens): 27x the cost of the answer itself
   50 answers a day: $  0.07 a day, $   1.98 a month for judging every one
  200 answers a day: $  0.26 a day, $   7.92 a month for judging every one
 1000 answers a day: $  1.32 a day, $  39.60 a month for judging every one
with the answer bank serving repeats: the model is not called, and the judge is not either


El juez cuesta varias veces lo que cuesta la respuesta, porque *piensa*. Sigue siendo barato por día, y dos
palancas lo mantienen así: juzgar una **muestra** en lugar de todo cuando crece el volumen, y dejar que el
**banco** absorba las preguntas repetidas (una respuesta del banco ni se genera ni se vuelve a juzgar). Los
precios de arriba son valores ilustrativos de un modelo alojado pequeño; toma los reales de la lista de precios
de tu proveedor.

## 9. Opcional: un juez real

Define `RAG_JUDGE_MODEL` con el nombre de un modelo de LiteLLM (un modelo de clase Haiku, o Amazon Nova Micro o
Lite a través de Bedrock; elige uno que **no** sea el modelo que escribe tus respuestas) y esta celda juzga el
primer turno con el prompt real, validando la respuesta con el mismo esquema Pydantic. **No se ejecuta en el
notebook guardado**: sin la variable no hace nada.

In [16]:
def litellm_judge(turn, index):
    import litellm
    prompt = JUDGE_PROMPT.format(pieces="\n".join(f"[{p['n']}] {p['text']}" for p in turn["pieces"]), question=turn["question"], answer=turn["answer"])
    r = litellm.completion(model=os.environ["RAG_JUDGE_MODEL"], messages=[{"role": "user", "content": prompt}],
                           max_tokens=3000, response_format={"type": "json_object"})     # generoso: el juez razona primero
    return json.loads(r.choices[0].message.content)

if os.getenv("RAG_JUDGE_MODEL"):
    print(judge(day1[0], index, judge_fn=litellm_judge).model_dump())
else:
    print("RAG_JUDGE_MODEL is not set: skipped (the notebook uses the deterministic stand-in judge).")

RAG_JUDGE_MODEL is not set: skipped (the notebook uses the deterministic stand-in judge).


## Cómo era el sistema real

Un asistente de previsión de cadena de suministro para comercio minorista ejecutaba este bucle cada día
laborable. Su recuperación era híbrida (BM25 y embeddings, fusionados con Reciprocal Rank Fusion), su banco de
respuestas se indexaba por un **umbral de coseno** de embeddings, y un juez (un modelo pequeño de clase Claude
a través de una pasarela, de una familia distinta a la del modelo que respondía) leía cada respuesta nueva por
la noche y escribía un veredicto, una puntuación de fidelidad, si las citas se sostenían y qué había fallado en
la recuperación. Una llamada al juez usaba unos **1.500 tokens de razonamiento**. Las correcciones de los
usuarios las juzgaba un segundo prompt y, cuando eran arreglos de documentación, abrían issues que un agente
programado convertía en un pull request por noche hacia una rama de desarrollo, **que el agente nunca
fusionaba**. Lo que leía el equipo era el resumen diario de fallos de recuperación, no la puntuación media.

## Ejercicios

1. Baja el umbral del banco de 0,95 a 0,75 y vuelve a ejecutar la sección 5. ¿A qué pregunta parecida empieza a
   servir una respuesta errónea? ¿Cuál es el coste de eso, comparado con las llamadas al modelo que ahorra?
2. El Día 2 no se movió ni con `boats` en `p02`. Elige los tres fallos del Día 2 que creas que una nota de
   vocabulario podría arreglar, añade las palabras a `APPROVED_EDITS` y vuelve a ejecutar la sección 7. El Día 2
   mejorará: ¿es el bucle generalizando o eres tú memorizando el Día 2? ¿Cómo lo distinguirías?
3. Escribe cinco observaciones más para la sección 6 (una fundada, una con una cifra que contradice el manual,
   una que pida un cambio de proceso) y comprueba adónde se encamina cada una.
4. En `stand_in_judge`, haz más estricta la regla de rechazo (`best_sim >= 0.35`). ¿Cómo cambian el número de
   fallos de recuperación y el kappa de la sección 8?
5. Anota cinco respuestas en las que un juez *modelo de lenguaje* se equivocaría con probabilidad. Para cada
   una, ¿qué etiqueta humana añadirías a la muestra semanal de calibración?